In [16]:
# library imports
library(stringr)
library(psych)
library(rjson)

In [36]:
# helper funcs

process_cla_string <- function(cla_string, outputted_type = 'numeric'){
  no_spaces <- str_replace_all(cla_string, ' ', '')
  fracs <- unlist(str_split(no_spaces, pattern = ';'))
  
  if(outputted_type == 'numeric'){
    fracs <- as.numeric(fracs)
  }
  else if((outputted_type == 'integer') | (outputted_type == 'int')){
    fracs <- as.integer(fracs)
  }
  
  return(fracs)
}



hky_sub_rate_mat <- function(frac_a, frac_g, frac_c, frac_t, transition_to_transversion_ratio = NULL,
                             baseline_transition_rate = NULL, baseline_transversion_rate = NULL){
  #' @title Hasegawa-Kishino-Yano (HKY) model of sequence evolution
  #' @description Assign mutation rates according to the HKY model of sequence evolution.
  #' @param frac_a numeric. The fraction of all nucleotides that are A
  #' @param frac_g numeric. The fraction of all nucleotides that are G
  #' @param frac_c numeric. The fraction of all nucleotides that are C
  #' @param frac_t numeric. The fraction of all nucleotides that are T
  #' @param transition_to_transversion_ratio numeric. The ratio of transition mutation rate to transversion mutation rate.
  #' @param baseline_transition_rate numeric. See details. Baseline transition rate that is further modified by to-base proportions in sequence.
  #' @param baseline_transversion_rate numeric. See details. Baseline transversion rate that is further modified by to-base proportions in sequence.
  #' @return 4x4 matrix containing substitution probabilities from A, G, C, T to A, G, C, T, respectively.
  #' @details HKY assumes variable base frequencies, one transition rate, and one transversion rate. 
  #' Note that mean_transition_rate and mean_transversion_rate are modified such that to-base substitution rates vary according to
  #' whether the to-base is a transition or transversion and according to to-base rates. 
  #' At least two of transition_to_transversion_ratio, baseline_transition_rate, and baseline_transversion_rate must be provided.
  #' Non-self substitution rates are scaled such that overall mean substitution rate across 12 non-self substitutions is equal to
  #' the harmonic mean of baseline_transition_rate and baseline_transversion_rate
  
  # rename
  ttv <- transition_to_transversion_ratio
  
  if(is.null(ttv)){
    ttv <- baseline_transition_rate / baseline_transversion_rate
  }
  if(is.null(baseline_transition_rate)){
    baseline_transition_rate <- ttv * baseline_transversion_rate
  }
  if(is.null(baseline_transversion_rate)){
    baseline_transversion_rate <- baseline_transition_rate / ttv
  }
  
  # account for both nucleotide composition of sequence as well as ttv
  # see notebook for formula derivation/logic
  from_a_rate <- baseline_transition_rate * c(0, ttv*(1+(frac_g - 0.25)), 1+(frac_c - 0.25), 1+(frac_t - 0.25))
  from_g_rate <- baseline_transition_rate * c(ttv*(1+(frac_g - 0.25)), 0, 1+(frac_c - 0.25), 1+(frac_t - 0.25))
  from_c_rate <- baseline_transition_rate * c(ttv*(1+(frac_g - 0.25)), 1+(frac_c - 0.25), 0, 1+(frac_t - 0.25))
  from_t_rate <- baseline_transition_rate * c(ttv*(1+(frac_g - 0.25)), 1+(frac_c - 0.25), 1+(frac_t - 0.25), 0)
  
  prenormalized_mat <- rbind(from_a_rate,
                             from_g_rate,
                             from_c_rate,
                             from_t_rate)
  
  harmonic_mean <- harmonic.mean(c(baseline_transition_rate, baseline_transversion_rate))
  
  prenorm_mean <- sum(prenormalized_mat) / 12
  scale_factor <- (harmonic_mean / prenorm_mean)
  norm_mat <- prenormalized_mat * scale_factor  
  
  return(norm_mat)
}


jc_sub_rate_mat <- function(overall_sub_rate){
  #' @title Jukes-Cantor model of sequence evolution
  #' @description Assign mutation rates according to the Jukes-Cantor model of sequence evolution.
  #' @param overall_sub_rate numeric. The overall probability of a substitution occurring.
  #' @return 4x4 matrix containing substitution probabilities from A, G, C, T to A, G, C, T, respectively. 
  #' @details JC assumes equal base frequencies and equally likely substitutions between all combinations of bases
  
  # equal rate for all substitutions
  
  # assumes equal nucleotide frequencies
  
  # sr <- overall_sub_rate
  
  rate_mat <- matrix(overall_sub_rate, nrow = 4, ncol = 4)
  diag(rate_mat) <- 0
  
  # rate_mat <- rbind(
  #   c(0, rep(sr, 3)),
  #   c(sr, 0, rep(sr, 2)),
  #   c(rep(sr, 2), 0, sr),
  #   c(rep(sr, 3), 0)
  # )
  
  return(rate_mat)
}


In [37]:
# input args to modified func:

set.seed(42)

raw_cla_submodel <- 'HKY'
selected_sub_model <- '3; 0.00000000003; 0.00000000001'
sequence_with_targets <- sample(c('A', 'G', 'C', 'T'), size = 100, replace = TRUE)


In [38]:
# funcs to be modified

parse_sub_model_params <- function(raw_cla_submodel, selected_sub_model, sequence_with_targets){
  split_mod_params <- process_cla_string(selected_sub_model)
  # print(paste0('raw_cla_submodel == ', raw_cla_submodel))
  # print(paste0('selected_sub_model == ', selected_sub_model))
  sub_model_params_list <- list()
  
  # print('split_mod_params == ')
  # print(split_mod_params)
  
  if(raw_cla_submodel == 'JC'){
    sub_model_params_list[['model_overall_subrate']] <- split_mod_params[1]
    sub_prob_mat <- jc_sub_rate_mat(overall_sub_rate = sub_model_params_list[['model_overall_subrate']])
  }
  
  if(raw_cla_submodel == 'K80'){
    sub_model_params_list[['transition_to_transversion_ratio']] <- split_mod_params[1]
    sub_model_params_list[['transition_rate']] <- split_mod_params[2]
    sub_model_params_list[['transversion_rate']] <- split_mod_params[3]
    
    sub_prob_mat <- k80_sub_rate_mat(transition_to_transversion_ratio = sub_model_params_list[['transition_to_transversion_ratio']],
                                     transition_rate = sub_model_params_list[['transition_rate']],
                                     transversion_rate = sub_model_params_list[['transversion_rate']])
  }
  
  if(raw_cla_submodel == 'K81'){
    # If K81: 'transition_rate; transversion_rate_weakstrong_conserved; transversion_rate_aminoketo_conserved'\n
    # If F81: 'baseline_overall_subrate' \n
    # If HKY: 'transition_to_transversion_ratio; baseline_transition_rate; baseline_transversion_rate'\n
    # If GTR: 'AG_rate; AC_rate; AT_rate; GC_rate; GT_rate; CT_rate'"),
    sub_model_params_list[['transition_rate']] <- split_mod_params[1]
    sub_model_params_list[['transverison_rate_weakstrong_conserved']] <- split_mod_params[2]
    sub_model_params_list[['transversion_rate_aminoketo_conserved']] <- split_mod_params[3]
    
    sub_prob_mat <- k81_sub_rate_mat(transition_rate = sub_model_params_list[['transition_rate']],
                                     transversion_rate_weakstrong_conserved = sub_model_params_list[['transverison_rate_weakstrong_conserved']],
                                     transversion_rate_aminoketo_conserved = sub_model_params_list[['transversion_rate_aminoketo_conserved']])
  }
  
  if(raw_cla_submodel == 'F81'){
    # calculate nucleotide fractions based on provided sequence with targets
    sub_model_params_list[['baseline_overall_subrate']] <- split_mod_params[1]
    
    nuc_counts <- table(sequence_with_targets)
    frac_a <- nuc_counts[['A']]
    frac_g <- nuc_counts[['G']]
    frac_c <- nuc_counts[['C']]
    frac_t <- nuc_counts[['T']]
    
    sub_model_params_list[['frac_a']] <- frac_a
    sub_model_params_list[['frac_g']] <- frac_g
    sub_model_params_list[['frac_c']] <- frac_c
    sub_model_params_list[['frac_t']] <- frac_t
    
    sub_prob_mat <- f81_sub_rate_mat(frac_a = sub_model_params_list[['frac_a']],
                                     frac_g = sub_model_params_list[['frac_g']],
                                     frac_c = sub_model_params_list[['frac_c']],
                                     frac_t = sub_model_params_list[['frac_t']],
                                     baseline_overall_sub_rate = sub_model_params_list[['baseline_overall_subrate']])
  }
  
  if(raw_cla_submodel == 'HKY'){
    sub_model_params_list[['transition_to_transversion_ratio']] <- split_mod_params[1]
    sub_model_params_list[['baseline_transition_rate']] <- split_mod_params[2]
    sub_model_params_list[['baseline_transversion_rate']] <- split_mod_params[3]
    
    # print(paste0('length(sequence_with_targets'))
    nuc_counts <- table(sequence_with_targets)
    frac_a <- nuc_counts[['A']]
    frac_g <- nuc_counts[['G']]
    frac_c <- nuc_counts[['C']]
    frac_t <- nuc_counts[['T']]
    
    sub_model_params_list[['frac_a']] <- frac_a
    sub_model_params_list[['frac_g']] <- frac_g
    sub_model_params_list[['frac_c']] <- frac_c
    sub_model_params_list[['frac_t']] <- frac_t
    
    sub_prob_mat <- hky_sub_rate_mat(frac_a = sub_model_params_list[['frac_a']],
                                     frac_g = sub_model_params_list[['frac_g']],
                                     frac_c = sub_model_params_list[['frac_c']],
                                     frac_t = sub_model_params_list[['frac_t']],
                                     transition_to_transversion_ratio = sub_model_params_list[['transition_to_transversion_ratio']],
                                     baseline_transition_rate = sub_model_params_list[['baseline_transition_rate']],
                                     baseline_transversion_rate = sub_model_params_list[['baseline_transversion_rate']])
  }
  
  if(raw_cla_submodel == 'GTR'){
    sub_model_params_list[['AG_rate']] <- split_mod_params[1]
    sub_model_params_list[['AC_rate']] <- split_mod_params[2]
    sub_model_params_list[['AT_rate']] <- split_mod_params[3]
    sub_model_params_list[['GC_rate']] <- split_mod_params[4]
    sub_model_params_list[['GT_rate']] <- split_mod_params[5]
    sub_model_params_list[['CT_rate']] <- split_mod_params[6]
    
    nuc_counts <- table(sequence_with_targets)
    frac_a <- nuc_counts[['A']]
    frac_g <- nuc_counts[['G']]
    frac_c <- nuc_counts[['C']]
    frac_t <- nuc_counts[['T']]
    
    sub_model_params_list[['frac_a']] <- frac_a
    sub_model_params_list[['frac_g']] <- frac_g
    sub_model_params_list[['frac_c']] <- frac_c
    sub_model_params_list[['frac_t']] <- frac_t
    
    sub_prob_mat <- gtr_sub_rate_mat(frac_a = sub_model_params_list[['frac_a']],
                                     frac_g = sub_model_params_list[['frac_g']],
                                     frac_c = sub_model_params_list[['frac_c']],
                                     frac_t = sub_model_params_list[['frac_t']],
                                     ag_rate = sub_model_params_list[['AG_rate']],
                                     ac_rate = sub_model_params_list[['AC_rate']],
                                     at_rate = sub_model_params_list[['AT_rate']],
                                     gc_rate = sub_model_params_list[['GC_rate']],
                                     gt_rate = sub_model_params_list[['GT_rate']],
                                     ct_rate = sub_model_params_list[['CT_rate']])
  }
  
  return_list <- list('sub_model_params_list' = sub_model_params_list,
                      'sub_prob_mat' = sub_prob_mat)
  
  return(return_list)
}

In [48]:
# testing outputs 

mt_sub_model_list <- parse_sub_model_params(raw_cla_submodel, selected_sub_model, sequence_with_targets)
mt_sub_model_params <- mt_sub_model_list[['sub_model_params_list']]
mt_sub_prob_mat <- mt_sub_model_list[['sub_prob_mat']]


In [49]:
mt_sub_model_list

from_a_rate,0.000000e+00,3.058011e-11,5.552486e-12,8.867403e-12
from_g_rate,3.058011e-11,0.000000e+00,5.552486e-12,8.867403e-12
from_c_rate,3.058011e-11,5.552486e-12,0.000000e+00,8.867403e-12
from_t_rate,3.058011e-11,5.552486e-12,8.867403e-12,0.000000e+00


In [40]:
mt_sub_model_list

from_a_rate,0.000000e+00,3.058011e-11,5.552486e-12,8.867403e-12
from_g_rate,3.058011e-11,0.000000e+00,5.552486e-12,8.867403e-12
from_c_rate,3.058011e-11,5.552486e-12,0.000000e+00,8.867403e-12
from_t_rate,3.058011e-11,5.552486e-12,8.867403e-12,0.000000e+00


In [31]:
# Read JSON file as a text string
json_text <- readLines('./sim5_params/nested_demo1.json', warn = FALSE)
json_string <- paste(json_text, collapse = "\n")

# Parse JSON
nested_args <- fromJSON(json_string)

In [32]:
nested_args

$num_init_cells
[1] 1

$sim_length
[1] "4:6:1"

$num_cores
[1] 10

$cell_cycle_length
[1] 1

$time_inc
[1] 0.5

$max_mito_genomes_per_cell
[1] "1"

$bc_length
[1] 300

$max_bc_ints_per_cell
[1] "1;3;10"

$variable_bc_ints_per_cell
NULL

$mito_genome_length
[1] 50

$be_target_config
NULL

$nuc_target_config
[1] "R"

$force_transversions
[1] TRUE

$be_targets
NULL

$be_editing_window
[1] 0

$be_decaying_window
[1] FALSE

$nuclease_targets
[1] "50:0.33;0.33;0.34"

$nuclease_editing_window
[1] 0

$nuclease_decaying_window
[1] TRUE

$bc_bg_indel_probs
[1] "0.00000000000000000000001; 0.00000000000000000000000003"

$mt_bg_indel_probs
[1] "0.0000000001; 0.0000000003"

$be_mutations_per_target_per_division
[1] 0.01

$nuc_insertions_per_target_per_division
[1] 0.01

$nuc_deletions_per_target_per_division
[1] 0.03

$savename
[1] "moretargetslowerrates"

$reconstruction_method
[1] "fasta_only"

$include_var_pos_fasta
[1] FALSE

$chosen_gamma
[1] "mt_gamma_site_model"

$score_approach
[1] "af; bin"

$fasta_type
[1] "terminal"

$sampling_fractions
[1] "0.8; 1"

$mt_allelic_fractions
[1] "0"

$bc_allelic_fractions
[1] "0"

$filter_binary_with_af
[1] TRUE

$mt_genome_recovery_prob
[1] 1

$bc_integration_recovery_prob
[1] "0.6;1"

$recon_modality
[1] "bc; mt"

$add_mito_jitter
[1] FALSE

$barcode_sequence
NULL

$plot_heatmaps
[1] FALSE

$be_conversion_pattern
[1] "C --> T"

$bc_nuc_composition
[1] "0.25; 0.25; 0.3; 0.2"

$cell_type_dict
$cell_type_dict$celltype1
$cell_type_dict$celltype1$substitution_model
[1] "HKY"

$cell_type_dict$celltype1$sub_model_params
[1] "3; 0.00000000003; 0.00000000001"


$cell_type_dict$celltype2
$cell_type_dict$celltype2$substitution_model
[1] "JC"

$cell_type_dict$celltype2$sub_model_params
[1] 0.01



$mt_substitution_model
[1] "HKY"

$mt_sub_model_params
[1] "3; 0.00000000003; 0.00000000001"

$mt_invariant_sites
[1] 0.05

$mt_nontarget_heterogeneity_gamma
[1] "0.5; 3; mean"

$bc_substitution_model
[1] "HKY"

$bc_sub_model_params
[1] "3; 0.000000000000000000003; 0.0000000000000000000001"

$bc_invariant_sites
[1] 0

$bc_nontarget_heterogeneity_gamma
[1] "0.5; 3; mean"

$jitter_fraction
[1] 0.05

In [ ]:
# testing outputs 

mt_sub_model_list <- parse_sub_model_params(raw_cla_submodel, selected_sub_model, sequence_with_targets)
mt_sub_model_params <- mt_sub_model_list[['sub_model_params_list']]
mt_sub_prob_mat <- mt_sub_model_list[['sub_prob_mat']]


In [33]:
nested_args$cell_type_dict[['celltype1']]

$substitution_model
[1] "HKY"

$sub_model_params
[1] "3; 0.00000000003; 0.00000000001"

In [47]:
mt_sub_model_list[['sub_model_params_list']]

$model_overall_subrate
[1] 0.01

In [45]:
growing_celltype_submodel_list <- list()
for(celltype in names(nested_args$cell_type_dict)){
    
    mt_sub_model_list <- parse_sub_model_params(raw_cla_submodel = nested_args$cell_type_dict[[celltype]]$substitution_model, 
                                                selected_sub_model = nested_args$cell_type_dict[[celltype]]$sub_model_params, 
                                                sequence_with_target = sequence_with_targets)
    growing_celltype_submodel_list[[celltype]] <- mt_sub_model_list
}

In [46]:
growing_celltype_submodel_list

$celltype1
$celltype1$sub_model_params_list
$celltype1$sub_model_params_list$transition_to_transversion_ratio
[1] 3

$celltype1$sub_model_params_list$baseline_transition_rate
[1] 3e-11

$celltype1$sub_model_params_list$baseline_transversion_rate
[1] 1e-11

$celltype1$sub_model_params_list$frac_a
[1] 28

$celltype1$sub_model_params_list$frac_g
[1] 30

$celltype1$sub_model_params_list$frac_c
[1] 16

$celltype1$sub_model_params_list$frac_t
[1] 26


$celltype1$sub_prob_mat
                    [,1]         [,2]         [,3]         [,4]
from_a_rate 0.000000e+00 3.058011e-11 5.552486e-12 8.867403e-12
from_g_rate 3.058011e-11 0.000000e+00 5.552486e-12 8.867403e-12
from_c_rate 3.058011e-11 5.552486e-12 0.000000e+00 8.867403e-12
from_t_rate 3.058011e-11 5.552486e-12 8.867403e-12 0.000000e+00


$celltype2
$celltype2$sub_model_params_list
$celltype2$sub_model_params_list$model_overall_subrate
[1] 0.01


$celltype2$sub_prob_mat
     [,1] [,2] [,3] [,4]
[1,] 0.00 0.01 0.01 0.01
[2,] 0.01 0.00 0.01 0.01
[3,] 0.01 0.01 0.00 0.01
[4,] 0.01 0.01 0.01 0.00

In [42]:
mt_sub_model_list

0.00,0.01,0.01,0.01
0.01,0.00,0.01,0.01
0.01,0.01,0.00,0.01
0.01,0.01,0.01,0.00


In [23]:
json_string

[1] "{\n\t\"num_init_cells\": 1,\n\t\"sim_length\": \"4:6:1\",\n\t\"num_cores\": 10,\n\t\"cell_cycle_length\": 1,\n\t\"time_inc\": 0.5,\n\t\"max_mito_genomes_per_cell\": \"1\",\n\t\"bc_length\": 300,\n\t\"max_bc_ints_per_cell\": \"1;3;10\",\n\t\"variable_bc_ints_per_cell\": null,\n\t\"mito_genome_length\": 50,\n\t\"be_target_config\": null,\n\t\"nuc_target_config\": \"R\",\n\t\"force_transversions\": true,\n\t\"be_targets\": null,\n\t\"be_editing_window\": 0,\n\t\"be_decaying_window\": false,\n\t\"nuclease_targets\": \"50:0.33;0.33;0.34\",\n\t\"nuclease_editing_window\": 0,\n\t\"nuclease_decaying_window\": true,\n\t\"bc_bg_indel_probs\": \"0.00000000000000000000001; 0.00000000000000000000000003\",\n\t\"mt_bg_indel_probs\": \"0.0000000001; 0.0000000003\",\n\t\"be_mutations_per_target_per_division\": 0.01,\n\t\"nuc_insertions_per_target_per_division\": 0.01,\n\t\"nuc_deletions_per_target_per_division\": 0.03,\n\t\"savename\": \"moretargetslowerrates\",\n\t\"reconstruction_method\": \"fasta_only\",\n\t\"include_var_pos_fasta\": false,\n\t\"chosen_gamma\": \"mt_gamma_site_model\",\n\t\"score_approach\": \"af; bin\",\n\t\"fasta_type\": \"terminal\",\n\t\"sampling_fractions\": \"0.8; 1\",\n\t\"mt_allelic_fractions\": \"0\",\n\t\"bc_allelic_fractions\": \"0\",\n\t\"filter_binary_with_af\": true,\n\t\"mt_genome_recovery_prob\": 1,\n\t\"bc_integration_recovery_prob\": \"0.6;1\",\n\t\"recon_modality\": \"bc; mt\",\n\t\"add_mito_jitter\": false,\n\t\"barcode_sequence\": null,\n\t\"plot_heatmaps\": false, \n\t\"be_conversion_pattern\": \"C --> T\",\n\t\"bc_nuc_composition\": \"0.25; 0.25; 0.3; 0.2\",\n\t\"cell_type_dict\": {\n\t\"celltype1\": {\n\t\"submodel\": \"HKY\",\n\t\"helperparams\":  \"3; 0.00000000003; 0.00000000001\"\n\t},\n\t\"celltype2\": {\n\t\"submodel\": \"JC\",\n\t\"helperparams\": \"testing1\"\n\t},\n\t},\n\t\"mt_substitution_model\": \"HKY\",\n\t\"mt_sub_model_params\": \"3; 0.00000000003; 0.00000000001\",\n\t\"mt_invariant_sites\": 0.05,\n\t\"mt_nontarget_heterogeneity_gamma\": \"0.5; 3; mean\",\n\t\"bc_substitution_model\": \"HKY\",\n\t\"bc_sub_model_params\": \"3; 0.000000000000000000003; 0.0000000000000000000001\",\n\t\"bc_invariant_sites\": 0,\n\t\"bc_nontarget_heterogeneity_gamma\": \"0.5; 3; mean\",\n\t\"jitter_fraction\": 0.05\n\n}\n\n\n"

In [25]:
# parse nested json:
nested <- fromJSON('./sim5_params/nested_demo1.json')

ERROR: Error in fromJSON("./sim5_params/nested_demo1.json"): not all data was parsed (0 chars were parsed out of a total of 31 chars)


In [ ]:
# i think for the downsampling integrations question, it comes down to whether you can msa multiple sequences 
# and output a fasta that is much longer than any of the individual sequences

# which intuitively would have to be true

# but if we were to coompletely drop out certain integrations from some cells, it would be necessary. 
# there should be very long gaps in each of the sequences corresponding to integration dropout

In [51]:
# to get certain cells to divide:
# we can't directly rep the mut profiles anymore
# we'll have a list of terminal cells
# and each will have a type
# do we select cells to die or divide first
# i think we probably select dying cells first
# we sample from the length according to weights specified by cell type death rates



# in a continuous cell state system, we ... 
# could do a weighted average of values between cell types
# eg if cell is transitinoing from type 1 to type 2 and val for type 1 is 1 and val for type 2 is 10,
# and the cell is 20% of the way to type 2, our new val would be 1(0.8)+10(0.2) = 2.8

# in a discrete system, allow each cell, at each timepoint or at each division point, to update its state according to a 
# markovian transition matrix
# in a continuous sytem, need a way to increase the probability of switching to type with increased progress toward that type 
# (assumption that development toward a cell type is unidirectional, but that doesn't need to be true)

# allow the cell state transition matrix to change during the simulation (i.e. update it at each timepoint)
# for example, what if we introduced some TF at some timepoint that leads to the development of one type of cell
# then our matrix would be different from that point onwards
# can pre-specify the pre-induced and post-induced matrices

# keep a running map of all cell names to cell types to be able to annotate a ground truth tree with colors == nodes
# can scale the colors according to how far along a cell is to a certain type in a continuous system
# check for concordance of this ground truth tree with the reconstructed tree

# eventually look to estimate the cell types of internal nodes? 
# you actually probably could do this. here's how:
# fix cell type transition values, cell type-specific division and death rates -- 
# or, fix them around certain values and add noise.
# run out the simulation without even caring about the mutational profiles; just ground truth tree topologies
# that inherently have some stochasticity because of the noise that surrounds these parameter estimates
# do this on simulations of different lengths and across different cell type transition vals/division/death rate combos

# we would observe different branching and death patterns in the tree at terminal timepoints
# bear in mind that each internal node of the ground truth trees will have some discrete or continuous reflection of cell type
# can you take inferred branch lengths and tree topology features and feed them into a classifier that estimates what internal node states are? 
# even, can we estimate the extent to which an internal node is comprised of each cell type? (more of a regressor question, or a class prob extension)

# the question we are able to solve is: how effectively can different combinations of lineage traicng technology parameters identify internal node cell types?
# eg vary the number of targets, the number of recovered integrations, etc. and make a reconstructed tree from each
# use these reconstructed features as the X_test, with our internal ndoes being y_test

# for each reconstructed tree, we would have a tree reconstruction accuracy as well as a cell type prediction auc (in class vs all out of class, e.g.)




In [ ]:
# pecking order:

# 1. check what happens when you msa sequences that are obviously missing integration(s). 
# validate: does muscle output a sequence that has length expected of ALL integrations concatenated?

# 2. allow for specification of cell-type specific mutation, division, and death rates.
# validate: do we see non-bifurcating trees in ground truth, and different mean numbers of edits/muts by cell type?

# 3. allow for specification of uninduced, induced_1, induced_2, ... induced_X which are transition matrices between cell types
# this will be another nested json deal. allow each transition matrix to be accompanied by the timepoint at which it takes hold

# 4. create training dataset by simulating lots of different ground truth trees
# remove all mutational processes from simulation? or could keep and store somewhere
# but what we're really interested in is just cell type data and tree dynamics (i.e. cell divisions and deaths)
# this might involve passing a param that says "enable mutations"; otherwise, could make length 1 barcodes/mt genomes. might be the easier play

# 5. feature extraction approach --> what parts of these ground truth trees enable us to accurately estimate internal node cell type?
# we will need to develop a feature vector associated with each internal node (things like depth (? could bias), num neighbors, branch lengths, etc.)
# can we derive branch lengths from a ground truth tree? maybe not
# look into features that other groups have used when X data is phylogenetic tree

# 6. construct crispr-informed trees that lead to reconstructed lineages, and feed these feature sets into our trained model
# determine which experimental and computational parameter combinations of lineage tracing experiments would allow us to 
# optimally predict internal node cell type. 

# 7. for a given set of lineage tracing experimental/computational factors, compare the lineage tree reconstruction accuracy
# to the internal cell type prediction accuracy